# EA3 - Actividad 3.1: Introduccion a Apache Kafka

## Objetivos
- Entender los conceptos fundamentales de mensajeria distribuida
- Crear topics en Apache Kafka
- Producir y consumir mensajes desde Python
- Trabajar con datos JSON estructurados en Kafka

> **IMPORTANTE:** Este notebook requiere el perfil **completo** de Docker.
> ```bash
> docker-compose --profile completo up -d
> ```
> Espera ~1 minuto para que Kafka y los demas servicios inicien completamente.

## Conceptos Clave

### Que es la Mensajeria Distribuida?

La mensajeria distribuida es un patron de arquitectura donde los **productores** envian mensajes a un sistema central (broker) y los **consumidores** los reciben de forma asincrona. Esto permite desacoplar los sistemas que generan datos de los que los procesan.

### Apache Kafka

Apache Kafka es una **plataforma de streaming distribuida** disenada para manejar flujos de datos en tiempo real con alta disponibilidad y durabilidad.

### Componentes Principales

```
  PRODUCERS                    KAFKA CLUSTER                    CONSUMERS
  =========                    =============                    =========

  +--------+                  +-- Topic: ventas --+            +----------+
  | App 1  |---send--->      | Particion 0: [m1][m2][m3] |---read--->| Consumer |
  +--------+                  | Particion 1: [m4][m5]     |           | Group A  |
                              | Particion 2: [m6][m7][m8] |           +----------+
  +--------+                  +----------------------------+
  | App 2  |---send--->                                        +----------+
  +--------+                  +-- Topic: logs ---+             | Consumer |
                              | Particion 0: [l1][l2]     |---read--->| Group B  |
                              | Particion 1: [l3]         |           +----------+
                              +----------------------------+
```

### Conceptos Fundamentales

| Concepto | Descripcion |
|----------|-------------|
| **Topic** | Canal con nombre donde se publican los mensajes. Similar a una "tabla" en una BD |
| **Particion** | Subdivision de un topic para paralelismo. Cada particion es una secuencia ordenada |
| **Offset** | Posicion unica de cada mensaje dentro de una particion |
| **Producer** | Aplicacion que envia (publica) mensajes a un topic |
| **Consumer** | Aplicacion que lee (consume) mensajes de un topic |
| **Consumer Group** | Grupo de consumers que se reparten la lectura de particiones |
| **Broker** | Servidor de Kafka que almacena y distribuye los mensajes |

### Flujo de un Mensaje

```
  Producer                Broker (Kafka)              Consumer
     |                        |                          |
     |--- send(topic, msg) -->|                          |
     |                        |-- almacena en           |
     |                        |   particion + offset --> |
     |                        |                          |
     |                        |<-- poll() --------------||
     |                        |--- entrega msg -------->|
     |                        |                          |
```

### Por que usar Kafka?

- **Alta throughput:** Puede manejar millones de mensajes por segundo
- **Durabilidad:** Los mensajes se persisten en disco y se replican
- **Escalabilidad:** Se puede escalar horizontalmente agregando brokers
- **Desacoplamiento:** Producers y consumers no necesitan conocerse entre si

## Setup

Importamos las librerias necesarias para trabajar con Kafka desde Python.

In [ ]:
import json
import time
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic

KAFKA_BROKER = "kafka:29092"
print("Librerias importadas correctamente")

## 1. Verificar Conexion con Kafka

Antes de trabajar con mensajes, verificamos que el broker de Kafka este disponible y funcionando.

In [ ]:
admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
print("Conexion exitosa!")
print(f"Topics existentes: {admin.list_topics()}")

## 2. Crear un Topic

Un **topic** es el canal donde se publican los mensajes. Podemos configurar:
- `num_partitions`: Cuantas particiones tendra (determina el paralelismo)
- `replication_factor`: Cuantas copias del dato se mantienen (alta disponibilidad)

In [ ]:
topic = NewTopic(name="test-bigdata", num_partitions=3, replication_factor=1)
try:
    admin.create_topics([topic])
    print("Topic 'test-bigdata' creado")
except Exception as e:
    print(f"Topic ya existe o error: {e}")

In [ ]:
# Verificar que el topic fue creado
print(f"Topics disponibles: {admin.list_topics()}")

## 3. Producir Mensajes Simples

Un **Producer** envia mensajes a un topic. Configuramos:
- `bootstrap_servers`: Direccion del broker
- `value_serializer`: Funcion para convertir el mensaje a bytes (Kafka solo maneja bytes)

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

for i in range(5):
    mensaje = {"id": i, "texto": f"Mensaje de prueba {i}"}
    producer.send("test-bigdata", value=mensaje)
    print(f"Enviado: {mensaje}")

producer.flush()
producer.close()
print("\nTodos los mensajes fueron enviados y el producer se cerro.")

## 4. Consumir Mensajes

Un **Consumer** lee mensajes de un topic. Configuramos:
- `auto_offset_reset='earliest'`: Leer desde el primer mensaje disponible
- `value_deserializer`: Funcion inversa al serializer del producer
- `consumer_timeout_ms`: Tiempo maximo de espera antes de detenerse si no llegan mensajes

In [ ]:
consumer = KafkaConsumer(
    "test-bigdata",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=5000
)

contador = 0
for mensaje in consumer:
    print(f"Recibido [particion={mensaje.partition}, offset={mensaje.offset}]: {mensaje.value}")
    contador += 1

consumer.close()
print(f"\nTotal mensajes consumidos: {contador}")

## 5. Enviar JSON Estructurado

En la practica, los mensajes de Kafka suelen ser **documentos JSON estructurados** que representan eventos del negocio (transacciones, logs, lecturas IoT, etc.).

Veamos un ejemplo con transacciones de ventas:

In [ ]:
import random
from datetime import datetime

# Crear un producer
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Datos de ejemplo
productos = ["Laptop", "Smartphone", "Tablet", "Monitor", "Teclado"]
metodos_pago = ["Tarjeta Credito", "Tarjeta Debito", "Efectivo", "Transferencia"]

# Crear topic para transacciones
try:
    topic_tx = NewTopic(name="transacciones-ejemplo", num_partitions=2, replication_factor=1)
    admin.create_topics([topic_tx])
    print("Topic 'transacciones-ejemplo' creado")
except Exception as e:
    print(f"Topic ya existe: {e}")

# Enviar transacciones
for i in range(10):
    transaccion = {
        "id": f"TX-{i:04d}",
        "timestamp": datetime.now().isoformat(),
        "producto": random.choice(productos),
        "monto": random.randint(5000, 500000),
        "metodo_pago": random.choice(metodos_pago),
        "tienda_id": random.randint(1, 5)
    }
    producer.send("transacciones-ejemplo", value=transaccion)
    print(f"Enviada: {transaccion['id']} - {transaccion['producto']} - ${transaccion['monto']:,}")

producer.flush()
producer.close()
print("\nTransacciones enviadas correctamente.")

In [ ]:
# Consumir las transacciones
consumer = KafkaConsumer(
    "transacciones-ejemplo",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=5000
)

print("Transacciones recibidas:")
print("-" * 60)
for msg in consumer:
    tx = msg.value
    print(f"{tx['id']} | {tx['producto']:12s} | ${tx['monto']:>10,} | {tx['metodo_pago']}")

consumer.close()

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Crear topic y producir mensajes
# =============================================================
# Creamos un topic "ventas-stream" y enviamos 10 transacciones JSON

from datetime import datetime

# 1. Crear topic
topic_ventas = NewTopic(name="ventas-stream", num_partitions=2, replication_factor=1)
try:
    admin.create_topics([topic_ventas])
    print("Topic 'ventas-stream' creado")
except Exception as e:
    print(f"Topic ya existe o error: {e}")

# 2. Crear producer con serializador JSON
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# 3. Enviar 10 transacciones
productos = ["Laptop", "Mouse", "Teclado", "Monitor", "Smartphone"]
for i in range(10):
    tx = {
        "id": f"VENTA-{i:04d}",
        "monto": random.randint(1000, 100000),
        "tienda_id": random.randint(1, 5),
        "producto": random.choice(productos),
        "timestamp": datetime.now().isoformat()
    }
    producer.send("ventas-stream", value=tx)
    print(f"Enviado: {tx}")

# 4. Flush y cerrar
producer.flush()
producer.close()
print(f"\n10 transacciones enviadas al topic 'ventas-stream'")

> **Nota docente:** La clave de este ejercicio es que los estudiantes repliquen el patron productor visto en la seccion guiada. Errores comunes: (1) olvidar `producer.flush()` antes de cerrar, lo que puede causar perdida de mensajes; (2) no manejar la excepcion al crear un topic que ya existe; (3) usar `value_serializer` incorrecto (debe devolver bytes). Recalcar que `json.dumps(v).encode('utf-8')` convierte dict -> JSON string -> bytes, que es lo que Kafka espera.

In [ ]:
# =============================================================
# EJERCICIO 2: Consumir y contar mensajes
# =============================================================
# Consumimos todos los mensajes del topic "ventas-stream"

consumer = KafkaConsumer(
    "ventas-stream",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=5000
)

count = 0
monto_total = 0
for msg in consumer:
    tx = msg.value
    print(f"Recibido [particion={msg.partition}, offset={msg.offset}]: {tx}")
    count += 1
    monto_total += tx.get("monto", 0)

consumer.close()
print(f"\nTotal mensajes recibidos: {count}")
print(f"Monto total: ${monto_total:,}")

> **Nota docente:** El parametro `auto_offset_reset='earliest'` es fundamental: sin el, un consumer nuevo solo leeria mensajes producidos *despues* de su suscripcion. El `consumer_timeout_ms=5000` evita que el consumer se quede bloqueado indefinidamente esperando mensajes. Error comun: olvidar `consumer.close()`, lo que puede dejar conexiones abiertas al broker. El monto total es un bonus que demuestra como procesar datos al consumir.

---
## Resumen

En esta actividad aprendimos:

1. **Mensajeria distribuida:** Patron donde producers y consumers se comunican a traves de un broker
2. **Componentes de Kafka:** Topics, particiones, offsets, producers, consumers y consumer groups
3. **Crear topics:** Usando `KafkaAdminClient` y `NewTopic`
4. **Producir mensajes:** Usando `KafkaProducer` con serializacion JSON
5. **Consumir mensajes:** Usando `KafkaConsumer` con deserializacion JSON
6. **JSON estructurado:** Enviar y recibir datos complejos como transacciones

---
## Desafio Extra (Opcional)

**Producer en tiempo real con consumo asincrono:**

Crea un producer que envie datos cada 2 segundos durante 20 segundos (un loop con `time.sleep(2)`). Luego, en la siguiente celda, consume todos los mensajes generados y verifica que se recibieron todos.

In [ ]:
# =============================================================
# DESAFIO: Producer en tiempo real
# =============================================================
# Enviamos datos cada 2 segundos simulando un stream en tiempo real

from datetime import datetime
import time

# Crear topic
try:
    topic_rt = NewTopic(name="stream-realtime", num_partitions=2, replication_factor=1)
    admin.create_topics([topic_rt])
    print("Topic 'stream-realtime' creado")
except Exception as e:
    print(f"Topic ya existe: {e}")

# Producer en tiempo real
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

for i in range(10):
    tx = {
        "id": f"live_{i:03d}",
        "monto": random.randint(1000, 100000),
        "producto": random.choice(["Laptop", "Mouse", "Teclado"]),
        "timestamp": datetime.now().isoformat()
    }
    producer.send("stream-realtime", value=tx)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Enviado: {tx}")
    time.sleep(2)

producer.flush()
producer.close()
print("\nProducer en tiempo real finalizado (10 mensajes en ~20 seg)")

> **Nota docente:** Este desafio introduce el concepto de produccion en tiempo real. El `time.sleep(2)` simula el intervalo entre eventos reales. En produccion, los eventos llegan naturalmente con intervalos variables. Punto de discusion: que pasa si el consumer se inicia *mientras* el producer esta enviando? Gracias a la persistencia de Kafka, el consumer puede leer tanto mensajes pasados como futuros.

In [ ]:
# =============================================================
# DESAFIO (continuacion): Consumir los mensajes del stream
# =============================================================
# Verificamos que se recibieron los 10 mensajes

consumer = KafkaConsumer(
    "stream-realtime",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=10000
)

count = 0
for msg in consumer:
    print(f"Recibido [{msg.partition}:{msg.offset}]: {msg.value}")
    count += 1

consumer.close()
print(f"\nTotal mensajes recibidos: {count}")
print(f"Verificacion: {'OK - Se recibieron los 10 mensajes' if count >= 10 else 'ATENCION - Se esperaban 10 mensajes'}") 

> **Nota docente:** Se usa `consumer_timeout_ms=10000` (10 segundos) en lugar de 5 porque los mensajes se enviaron con delay y podrian seguir llegando. La verificacion al final refuerza la importancia de confirmar que los datos fueron transmitidos correctamente, un patron comun en sistemas distribuidos.